# Modul 10: Regressions- und Klassifikationsmodelle vergleichen | Lösungen

## Überblick

Sie vergleichen repräsentative lineare, regularisierte, robuste, nachbarschaftsbasierte, baumbasierte, Ensemble- und SVM-Modelle. Einheitliche Splits, Vorverarbeitung, Baselines, mehrere Metriken, Laufzeiten, Residuen, Lernkurven, Klassenungleichgewicht und Voting sichern faire Vergleiche.

**Zugehörige Vorlesungen**

- **Regression vergleichen**
- **Klassifikation vergleichen**

## Lernziele

Nach der Bearbeitung können Sie:

- lineare, regularisierte und nichtlineare Regressoren in vergleichbaren Pipelines trainieren und bewerten.
- lineare, probabilistische, baumbasierte, Ensemble- und SVM-Klassifikatoren fair vergleichen.
- Modellwahl mit Metriken, Residuen, Lernkurven, Klassen- beziehungsweise Stichprobengewichten und Laufzeit begründen.

## Geprüfte Fähigkeiten

- MAE, RMSE, R², Residuen und Lernkurven für Regression
- Accuracy, Balanced Accuracy, Konfusionsmatrix und Fehlerarten für Klassifikation
- Ridge, Lasso, k-NN, Bäume, Boosting, SVM und Voting unter gleichen Bedingungen

## Hinweise zur Bearbeitung

Dieses Lösungsnotebook enthält dieselben Aufgaben wie das Übungsnotebook sowie vollständige, ausführlich kommentierte Musterlösungen. Bearbeiten Sie nach Möglichkeit zuerst das Übungsnotebook und nutzen Sie dieses Dokument anschließend zur Kontrolle und Vertiefung.

- **Erwarteter Schwierigkeitsgrad:** mittel bis anspruchsvoll
- Verwenden Sie sprechende Variablennamen und prüfen Sie wichtige Zwischenformen und Wertebereiche.
- Verändern Sie die vorgegebenen Zufalls-Startwerte nur, wenn eine Aufgabe dies ausdrücklich verlangt.
- Interpretieren Sie Ergebnisse fachlich. Eine einzelne Kennzahl ist selten eine vollständige Begründung.
- Alle Aufgaben sind für die kostenlose Google-Colab-Umgebung ausgelegt. Die Datensätze und Modelle sind bewusst klein gehalten. Eine GPU ist nicht erforderlich, kann aber bei einzelnen Deep-Learning-Aufgaben die Laufzeit verkürzen.

## Einrichtung und gemeinsame Datenbasis

Für Regression wird der kleine Diabetes-Datensatz verwendet. Für Klassifikation entstehen reproduzierbare synthetische Daten, darunter eine unausgewogene Variante. Suchräume und Modellgrößen bleiben klein.

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes, make_classification, make_moons
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import AdaBoostClassifier, HistGradientBoostingRegressor, RandomForestClassifier, VotingClassifier
from sklearn.linear_model import Lasso, LinearRegression, LogisticRegression, Ridge, SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import learning_curve, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

RANDOM_SEED = 42

# Regressionsdaten.
diabetes = load_diabetes()
X_reg, y_reg = diabetes.data, diabetes.target
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg,
    y_reg,
    test_size=0.25,
    random_state=RANDOM_SEED,
)

# Moderat nichtlineare Klassifikationsdaten.
X_cls, y_cls = make_moons(n_samples=500, noise=0.22, random_state=RANDOM_SEED)
X_cls_train, X_cls_test, y_cls_train, y_cls_test = train_test_split(
    X_cls,
    y_cls,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=y_cls,
)

# Unausgewogene Klassifikationsdaten für Gewichte.
X_imb, y_imb = make_classification(
    n_samples=700,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    weights=[0.88, 0.12],
    flip_y=0.02,
    random_state=RANDOM_SEED,
)
X_imb_train, X_imb_test, y_imb_train, y_imb_test = train_test_split(
    X_imb,
    y_imb,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=y_imb,
)

print("Einrichtung abgeschlossen.")
print("Regression Train/Test:", X_reg_train.shape, X_reg_test.shape)
print("Klassifikation Train/Test:", X_cls_train.shape, X_cls_test.shape)

### Aufgabe 1: Lineare und regularisierte Regression vergleichen

Vergleichen Sie auf exakt demselben Regressionssplit:

- Mittelwert-Baseline,
- lineare Regression,
- Ridge mit `alpha=1.0`,
- Lasso mit `alpha=0.1`.

Verwenden Sie bei Ridge und Lasso eine Skalierungs-Pipeline. Berechnen Sie MAE, RMSE, R² und Trainingszeit. Zählen Sie bei Lasso zusätzlich Koeffizienten, die praktisch null sind.

In [ ]:
def regressionsmetriken(y_true, y_pred):
    """Liefert MAE, RMSE und R²."""
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def regressionsmetriken(y_true, y_pred):
    """Liefert MAE, RMSE und R²."""
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }


reg_modelle = {
    "Mittelwert-Baseline": DummyRegressor(strategy="mean"),
    "Lineare Regression": LinearRegression(),
    "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Lasso": make_pipeline(StandardScaler(), Lasso(alpha=0.1, max_iter=10000, random_state=RANDOM_SEED)),
}

reg_ergebnisse = []
reg_vorhersagen = {}
for name, modell in reg_modelle.items():
    start = time.perf_counter()
    modell.fit(X_reg_train, y_reg_train)
    dauer = time.perf_counter() - start
    pred = modell.predict(X_reg_test)
    reg_vorhersagen[name] = pred
    zeile = {"Modell": name, "Trainingszeit_s": dauer, **regressionsmetriken(y_reg_test, pred)}

    if name == "Lasso":
        lasso_modell = modell.named_steps["lasso"]
        zeile["Nahe_Null_Koeffizienten"] = int(np.sum(np.abs(lasso_modell.coef_) < 1e-8))
    else:
        zeile["Nahe_Null_Koeffizienten"] = np.nan
    reg_ergebnisse.append(zeile)

reg_vergleich = pd.DataFrame(reg_ergebnisse).sort_values("RMSE")
display(reg_vergleich.round(4))

> **Musterantwort und Interpretation**
>
> Ridge verkleinert Koeffizienten kontinuierlich und stabilisiert Modelle bei korrelierten Merkmalen. Lasso kann einige Koeffizienten exakt oder nahezu null setzen und dadurch eine Form der Merkmalsauswahl bewirken. Die Regularisierungsstärke muss validiert werden.

### Aufgabe 2: Nichtlineare Regressoren und robuste Modellwahl

Vergleichen Sie zusätzlich:

- Polynommerkmale Grad 2 mit Ridge,
- k-NN-Regressor,
- Entscheidungsbaum mit begrenzter Tiefe,
- HistGradientBoostingRegressor,
- SVR mit RBF-Kernel.

Verwenden Sie passende Pipelines, dieselben Testdaten und die Metrikfunktion aus Aufgabe 1. Halten Sie alle Modelle klein und reproduzierbar.

In [ ]:
# Ergänzen Sie die Tabelle reg_vergleich aus Aufgabe 1 um nichtlineare Modelle.

# ============================================================
# MUSTERLÖSUNG
# ============================================================

nichtlineare_modelle = {
    "Polynom Grad 2 + Ridge": make_pipeline(
        PolynomialFeatures(degree=2, include_bias=False),
        StandardScaler(),
        Ridge(alpha=3.0),
    ),
    "k-NN Regression": make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=9)),
    "Entscheidungsbaum": DecisionTreeRegressor(max_depth=4, min_samples_leaf=8, random_state=RANDOM_SEED),
    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=120,
        max_leaf_nodes=12,
        learning_rate=0.06,
        random_state=RANDOM_SEED,
    ),
    "SVR RBF": make_pipeline(StandardScaler(), SVR(C=30.0, epsilon=10.0, gamma="scale")),
}

weitere_reg_ergebnisse = []
for name, modell in nichtlineare_modelle.items():
    start = time.perf_counter()
    modell.fit(X_reg_train, y_reg_train)
    dauer = time.perf_counter() - start
    pred = modell.predict(X_reg_test)
    reg_vorhersagen[name] = pred
    weitere_reg_ergebnisse.append(
        {"Modell": name, "Trainingszeit_s": dauer, **regressionsmetriken(y_reg_test, pred)}
    )

reg_gesamt = pd.concat(
    [reg_vergleich.drop(columns=["Nahe_Null_Koeffizienten"]), pd.DataFrame(weitere_reg_ergebnisse)],
    ignore_index=True,
).sort_values("RMSE")
display(reg_gesamt.round(4))

> **Musterantwort und Interpretation**
>
> Ein einzelner Split unterliegt Zufallsschwankungen und kann eine Modellklasse bevorzugen. Zusätzlich zählen Stabilität, Laufzeit, Interpretierbarkeit, Datenmenge, Residuenmuster und Einsatzanforderungen. Hyperparameter sollten auf Validierung oder Kreuzvalidierung gewählt und der Test erst am Ende verwendet werden.

### Aufgabe 3: Residualanalyse und Lernkurve für Regression

1. Wählen Sie das Modell mit dem kleinsten Test-RMSE aus `reg_gesamt` und stellen Sie Residuen gegen Vorhersagen dar.
2. Markieren Sie die fünf größten absoluten Residuen.
3. Erzeugen Sie für eine Ridge-Pipeline eine Lernkurve mit fünf Trainingsgrößen und drei Folds.
4. Visualisieren Sie Trainings- und Validierungs-RMSE.
5. Interpretieren Sie Bias, Varianz und Nutzen zusätzlicher Daten vorsichtig.

In [ ]:
# Die Modellobjekte liegen in reg_modelle und nichtlineare_modelle; Vorhersagen in reg_vorhersagen.

# ============================================================
# MUSTERLÖSUNG
# ============================================================

bester_name = reg_gesamt.iloc[0]["Modell"]
beste_pred = reg_vorhersagen[bester_name]
residuen = beste_pred - y_reg_test
abs_residuen = np.abs(residuen)
groesste_idx = np.argsort(abs_residuen)[-5:]

fig, ax = plt.subplots(figsize=(8, 5))
ax.axhline(0, linewidth=1)
ax.scatter(beste_pred, residuen, alpha=0.7)
ax.scatter(beste_pred[groesste_idx], residuen[groesste_idx], s=90, label="fünf größte |Residuen|")
ax.set_title(f"Residualanalyse: {bester_name}")
ax.set_xlabel("Vorhersage")
ax.set_ylabel("Vorhersage minus Istwert")
ax.legend()
plt.show()

ridge_lernmodell = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
train_sizes, train_scores, valid_scores = learning_curve(
    ridge_lernmodell,
    X_reg_train,
    y_reg_train,
    train_sizes=np.linspace(0.2, 1.0, 5),
    cv=3,
    scoring="neg_root_mean_squared_error",
    shuffle=True,
    random_state=RANDOM_SEED,
)

train_rmse = -train_scores.mean(axis=1)
valid_rmse = -valid_scores.mean(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(train_sizes, train_rmse, marker="o", label="Training")
ax.plot(train_sizes, valid_rmse, marker="o", label="Validierung")
ax.set_title("Lernkurve der Ridge-Pipeline")
ax.set_xlabel("Trainingsbeispiele")
ax.set_ylabel("RMSE")
ax.legend()
plt.show()

> **Musterantwort und Interpretation**
>
> Eine große Lücke mit deutlich besserem Training kann auf hohe Varianz beziehungsweise Überanpassung hinweisen. Mehr passende Daten, stärkere Regularisierung oder ein einfacheres Modell könnten helfen. Die Interpretation muss jedoch Streuung zwischen Folds und Datenqualität berücksichtigen.

### Aufgabe 4: Klassifikatoren auf nichtlinearen Daten vergleichen

Vergleichen Sie auf dem Moon-Split:

- logistische Regression,
- SGDClassifier mit Log-Loss,
- GaussianNB,
- Entscheidungsbaum,
- Random Forest,
- AdaBoost,
- RBF-SVM mit Wahrscheinlichkeiten.

Verwenden Sie Skalierung für die linearen und SVM-Modelle. Berechnen Sie Accuracy, Balanced Accuracy und Trainingszeit. Visualisieren Sie die Entscheidungsgrenzen der logistischen Regression, des Baums und der RBF-SVM.

In [ ]:
def klassifikationsmetriken(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, y_pred),
    }

# ============================================================
# MUSTERLÖSUNG
# ============================================================

cls_modelle = {
    "Logistische Regression": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
    ),
    "SGD Log-Loss": make_pipeline(
        StandardScaler(),
        SGDClassifier(loss="log_loss", max_iter=2000, tol=1e-4, random_state=RANDOM_SEED),
    ),
    "GaussianNB": GaussianNB(),
    "Entscheidungsbaum": DecisionTreeClassifier(max_depth=4, min_samples_leaf=6, random_state=RANDOM_SEED),
    "Random Forest": RandomForestClassifier(
        n_estimators=120, max_depth=6, min_samples_leaf=3, random_state=RANDOM_SEED, n_jobs=-1
    ),
    "AdaBoost": AdaBoostClassifier(n_estimators=80, learning_rate=0.6, random_state=RANDOM_SEED),
    "RBF-SVM": make_pipeline(StandardScaler(), SVC(C=2.0, gamma="scale", probability=True, random_state=RANDOM_SEED)),
}

cls_ergebnisse = []
for name, modell in cls_modelle.items():
    start = time.perf_counter()
    modell.fit(X_cls_train, y_cls_train)
    dauer = time.perf_counter() - start
    pred = modell.predict(X_cls_test)
    cls_ergebnisse.append({"Modell": name, "Trainingszeit_s": dauer, **klassifikationsmetriken(y_cls_test, pred)})

cls_vergleich = pd.DataFrame(cls_ergebnisse).sort_values("Balanced_Accuracy", ascending=False)
display(cls_vergleich.round(4))

# Gemeinsames Raster für drei anschauliche Entscheidungsgrenzen.
x_min, x_max = X_cls[:, 0].min() - 0.6, X_cls[:, 0].max() + 0.6
y_min, y_max = X_cls[:, 1].min() - 0.6, X_cls[:, 1].max() + 0.6
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 220), np.linspace(y_min, y_max, 220))
raster = np.column_stack([xx.ravel(), yy.ravel()])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes, ["Logistische Regression", "Entscheidungsbaum", "RBF-SVM"]):
    modell = cls_modelle[name]
    raster_pred = modell.predict(raster).reshape(xx.shape)
    ax.contourf(xx, yy, raster_pred, alpha=0.25)
    ax.scatter(X_cls_test[:, 0], X_cls_test[:, 1], c=y_cls_test, edgecolor="black", s=25)
    ax.set_title(name)
    ax.set_xlabel("Merkmal 1")
    ax.set_ylabel("Merkmal 2")
plt.tight_layout()
plt.show()

> **Musterantwort und Interpretation**
>
> Skalierung verändert Größenordnungen, aber nicht die grundsätzliche Form der Entscheidungsgrenze. Eine lineare logistische Regression kann die gekrümmten, ineinandergreifenden Gruppen nur mit einer Geraden trennen. Nichtlineare Modelle können diese Geometrie flexibler abbilden.

### Aufgabe 5: Klassenungleichgewicht, Gewichte und Konfusionsmatrix

Trainieren Sie auf den unausgewogenen Daten zwei logistische Pipelines:

- ohne Klassengewicht,
- mit `class_weight="balanced"`.

Vergleichen Sie Accuracy, Balanced Accuracy, Konfusionsmatrix, Recall der positiven Klasse und Anzahl der Fehlalarme. Erklären Sie den Zielkonflikt.

In [ ]:
def positive_recall_und_fp(y_true, y_pred):
    matrix = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = matrix.ravel()
    recall_pos = tp / (tp + fn) if (tp + fn) else 0.0
    return matrix, recall_pos, fp

# ============================================================
# MUSTERLÖSUNG
# ============================================================

imb_modelle = {
    "ohne Gewicht": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
    ),
    "balanced": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_SEED),
    ),
}

imb_zeilen = []
for name, modell in imb_modelle.items():
    modell.fit(X_imb_train, y_imb_train)
    pred = modell.predict(X_imb_test)
    matrix, recall_pos, fp = positive_recall_und_fp(y_imb_test, pred)
    imb_zeilen.append(
        {
            "Modell": name,
            "Accuracy": accuracy_score(y_imb_test, pred),
            "Balanced_Accuracy": balanced_accuracy_score(y_imb_test, pred),
            "Recall_Positive": recall_pos,
            "Fehlalarme_FP": fp,
            "Konfusionsmatrix": matrix.tolist(),
        }
    )

imb_vergleich = pd.DataFrame(imb_zeilen)
display(imb_vergleich)

> **Musterantwort und Interpretation**
>
> Sie ist sinnvoll, wenn das Übersehen positiver Fälle deutlich schwerere Folgen hat als eine zusätzliche Prüfung negativer Fälle. Die Entscheidung sollte auf konkreten Fehlerkosten und einer geeigneten Schwelle beruhen, nicht nur auf einer allgemeinen Präferenz für höheren Recall.

### Aufgabe 6: Voting-Ensemble und gemeinsame Fehleranalyse

Bauen Sie auf den Moon-Daten ein Soft-Voting-Ensemble aus:

- logistischer Regression,
- Random Forest,
- RBF-SVM mit Wahrscheinlichkeiten.

1. Trainieren Sie alle Einzelmodelle und das Voting-Modell auf demselben Split.
2. Vergleichen Sie Accuracy und Balanced Accuracy.
3. Finden Sie Testpunkte, bei denen die Einzelmodelle uneinig sind.
4. Zeigen Sie die vorhergesagten Wahrscheinlichkeiten dieser Punkte.
5. Beurteilen Sie, ob Voting automatisch besser sein muss.

In [ ]:
log_vote = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=RANDOM_SEED))
rf_vote = RandomForestClassifier(n_estimators=120, max_depth=6, random_state=RANDOM_SEED, n_jobs=-1)
svm_vote = make_pipeline(StandardScaler(), SVC(C=2.0, probability=True, random_state=RANDOM_SEED))

# ============================================================
# MUSTERLÖSUNG
# ============================================================

voting = VotingClassifier(
    estimators=[("log", log_vote), ("rf", rf_vote), ("svm", svm_vote)],
    voting="soft",
)

vote_modelle = {
    "Logistisch": log_vote,
    "Random Forest": rf_vote,
    "RBF-SVM": svm_vote,
    "Soft Voting": voting,
}

vote_preds = {}
vote_probs = {}
vote_ergebnisse = []
for name, modell in vote_modelle.items():
    modell.fit(X_cls_train, y_cls_train)
    pred = modell.predict(X_cls_test)
    vote_preds[name] = pred
    vote_probs[name] = modell.predict_proba(X_cls_test)[:, 1]
    vote_ergebnisse.append({"Modell": name, **klassifikationsmetriken(y_cls_test, pred)})

display(pd.DataFrame(vote_ergebnisse).sort_values("Balanced_Accuracy", ascending=False).round(3))

# Uneinigkeit liegt vor, wenn nicht alle drei Einzelmodelle dasselbe Label liefern.
einzelmatrix = np.column_stack(
    [vote_preds["Logistisch"], vote_preds["Random Forest"], vote_preds["RBF-SVM"]]
)
uneinig = np.any(einzelmatrix != einzelmatrix[:, [0]], axis=1)
uneinig_idx = np.where(uneinig)[0]

analyse_uneinig = pd.DataFrame(
    {
        "Ist": y_cls_test[uneinig_idx],
        "P_log": vote_probs["Logistisch"][uneinig_idx],
        "P_rf": vote_probs["Random Forest"][uneinig_idx],
        "P_svm": vote_probs["RBF-SVM"][uneinig_idx],
        "P_voting": vote_probs["Soft Voting"][uneinig_idx],
    }
)
print("Uneinige Testpunkte:", len(analyse_uneinig))
display(analyse_uneinig.head(10).round(3))

> **Musterantwort und Interpretation**
>
> Voting mittelt auch die Fehler und schlecht kalibrierten Wahrscheinlichkeiten schwächerer Modelle. Wenn die Modelle sehr ähnlich sind oder ein Modell systematisch irrt, entsteht wenig nützliche Vielfalt. Gewichte und Modellwahl müssen validiert werden, und das Ensemble erhöht Komplexität sowie Wartungsaufwand.

### Aufgabe 7: Integrationsaufgabe: Modellentscheidung unter Praxisbedingungen

Erstellen Sie zwei Entscheidungstabellen:

1. **Regression:** Empfehlen Sie ein Modell für eine Anwendung, in der MAE verständlich, Inferenz schnell und Interpretierbarkeit wichtig ist.
2. **Klassifikation:** Empfehlen Sie ein Modell für gekrümmte Grenzen, aber begrenzte Laufzeit und nachvollziehbare Fehleranalyse.

Nutzen Sie Ihre gemessenen Metriken und Laufzeiten. Dokumentieren Sie außerdem Baseline, Split, wichtigste Grenze und einen nächsten Validierungsschritt.

In [ ]:
# Verwenden Sie reg_gesamt, cls_vergleich und die Analysen der vorherigen Aufgaben.

# ============================================================
# MUSTERLÖSUNG
# ============================================================

reg_empfehlung = pd.DataFrame(
    {
        "Kriterium": ["Empfohlenes Modell", "Begründung", "Baseline", "Wichtigste Grenze", "Nächster Schritt"],
        "Bewertung": [
            "Ridge oder lineare Regression, abhängig vom gemessenen Fehlerunterschied",
            "Beide sind schnell und vergleichsweise interpretierbar; Ridge stabilisiert korrelierte Merkmale.",
            "Mittelwert-DummyRegressor",
            "Lineare Struktur kann relevante Nichtlinearitäten oder Teilgruppenfehler übersehen.",
            "Regularisierungsstärke mit leakage-sicherer Kreuzvalidierung prüfen und Residuen nach Teilgruppen analysieren.",
        ],
    }
)

cls_empfehlung = pd.DataFrame(
    {
        "Kriterium": ["Empfohlenes Modell", "Begründung", "Baseline", "Wichtigste Grenze", "Nächster Schritt"],
        "Bewertung": [
            "Begrenzter Entscheidungsbaum oder kleiner Random Forest",
            "Erfasst gekrümmte Grenzen; Baum ist direkt visualisierbar, Forest meist stabiler.",
            "Mehrheitsklassifikator",
            "Baum kann instabil sein; Forest ist weniger transparent und benötigt mehr Inferenzaufwand.",
            "Tiefe und Mindestblattgröße validieren, Konfusionsmatrix und Schwellen nach Fehlerkosten prüfen.",
        ],
    }
)

display(reg_empfehlung)
display(cls_empfehlung)

> **Musterantwort und Interpretation**
>
> Die beste Wahl ist nicht automatisch das Modell mit dem kleinsten einzelnen Testfehler. Für die Regression ist ein lineares oder Ridge-Modell plausibel, wenn sein Fehler nahe am Bestwert liegt und Interpretierbarkeit sowie Geschwindigkeit wichtig sind. Für die nichtlineare Klassifikation bietet ein begrenzter Baum hohe Verständlichkeit, während ein kleiner Forest meist stabiler ist. Beide Entscheidungen müssen mit Validierung, Fehlerkosten und Einsatzdaten abgesichert werden.

## Abschlusskontrolle

Prüfen Sie vor dem Abschluss:

- Lassen sich alle Zellen in sinnvoller Reihenfolge ausführen?
- Sind Formen, Datentypen, Wertebereiche und Zufalls-Startwerte dokumentiert?
- Wurden Trainings-, Validierungs- und Testinformationen sauber getrennt?
- Sind Diagramme und Kennzahlen beschriftet und fachlich interpretiert?
- Können Sie erklären, warum die gewählten Methoden zur Aufgabenstellung passen?